> 在线讲义：[第 18 章 · Scaling law 与全生命周期成本：训练多大、训练多久、服务多少次](https://wikieden.github.io/ai-infra-from-constraints/lessons/18-scaling-lifecycle.html) · 本书《从约束推导 AI Infra》

# 第 18 章 · Scaling law 与全生命周期成本：训练多大、训练多久、服务多少次

配套讲义：仓库里 `ai-infra/notes/lessons/18-scaling-lifecycle.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 预算分配 | 指数 | `alloc_exponents(alpha, beta)` | $\alpha = \beta$ 时各 0.5 |
| 第二步 留出检验 | 均方根 | `rmse([2.583, 2.363], [2.574, 2.337])` | 0.0195（原文用未取整的值得 0.0193） |
| 第三步 $D/N$ 与 $6ND$ | 除法、乘法 | `history()` | 149 … 4500；$4.02 \times 10^{22}$ … |
| 第三步 V4-Flash 专家 | $18 n_L h f \cdot 7$ | `expert_train_flops(43, 4096, 2048, 6, 1)` | 约 45.45 GFLOPs/token |
| 第四步 工期 | 除以卡数 | `calendar_days(1_022_362, 2048)` 等 | 20.80、54.20、78.43 天 |
| 第五步 回本 | $\Delta C_0 / \Delta c$ | `breakeven_tasks(1e5 * 3600, 0.72)` | $5 \times 10^8$ |

例题 3-5 的留出点、Llama / Qwen 训练投入表、GPU 小时、回本例子取自 AII 第 3 章 §3.6。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
HISTORY = [
    ("Llama 1 6.7B", 6.7e9, 1e12),
    ("Llama 2 7B", 7e9, 2e12),
    ("Llama 3.1 8B", 8e9, 15e12),
    ("Qwen2.5 7B", 7e9, 18e12),
    ("Qwen3 8B", 8e9, 36e12),
]

**函数 `alloc_exponents`**：最优分配的指数：N_opt ∝ C^(β/(α+β))，D_opt ∝ C^(α/(α+β))。

In [ ]:
def alloc_exponents(alpha: float, beta: float) -> tuple[float, float]:
    """最优分配的指数：N_opt ∝ C^(β/(α+β))，D_opt ∝ C^(α/(α+β))。"""
    return beta / (alpha + beta), alpha / (alpha + beta)

**函数 `rmse`**：均方根误差。

In [ ]:
def rmse(pred: list[float], actual: list[float]) -> float:
    """均方根误差。"""
    return math.sqrt(sum((p - a) ** 2 for p, a in zip(pred, actual)) / len(pred))

**函数 `history`**：(模型, D/N, 6ND)。

In [ ]:
def history() -> list[tuple[str, float, float]]:
    """(模型, D/N, 6ND)。"""
    return [(name, d / n, 6 * n * d) for name, n, d in HISTORY]

**函数 `expert_train_flops`**：MoE 主干专家每 token 的训练运算：18 · n_L · h · f · (k_r + k_s)。

In [ ]:
def expert_train_flops(n_layers: int, h: int, f: int, k_routed: int, k_shared: int) -> float:
    """MoE 主干专家每 token 的训练运算：18 · n_L · h · f · (k_r + k_s)。"""
    return 18 * n_layers * h * f * (k_routed + k_shared)

**函数 `calendar_days`**：工期（天）= GPU 小时 ÷ 卡数 ÷ 24。

In [ ]:
def calendar_days(gpu_hours: float, n_gpu: int) -> float:
    """工期（天）= GPU 小时 ÷ 卡数 ÷ 24。"""
    return gpu_hours / n_gpu / 24

**函数 `breakeven_tasks`**：多花的训练要多少次任务才回本：ΔC0 / Δc。

In [ ]:
def breakeven_tasks(extra_train_gpu_s: float, saving_per_task_gpu_s: float) -> float:
    """多花的训练要多少次任务才回本：ΔC0 / Δc。"""
    return extra_train_gpu_s / saving_per_task_gpu_s

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：α = β 时的分配指数")
print(f"  N、D 各 ∝ C^{alloc_exponents(0.3, 0.3)[0]}：预算 ×4 → 各 ×{4 ** 0.5:.0f}")

In [ ]:
print("\n第二步：例题 3-5 的两个留出点")
print(f"  均方根误差 {rmse([2.583, 2.363], [2.574, 2.337]):.4f}（原文用未取整的值得 0.0193）")

In [ ]:
print("\n第三步：相近规模模型的训练投入")
for name, dn, c in history():
    print(f"  {name:13s} D/N ≈ {dn:7,.0f}   6ND ≈ {c:.3g}")
print(f"  V4-Flash 主干专家：{expert_train_flops(43, 4096, 2048, 6, 1) / 1e9:.2f} GFLOPs/token")

In [ ]:
print("\n第四步：GPU 小时 → 工期")
for name, h, n in (("Llama 1 65B", 1_022_362, 2048), ("DeepSeek-V3 预训练", 2.664e6, 2048), ("Llama 3.1 405B", 30.84e6, 16384)):
    print(f"  {name}：{calendar_days(h, n):.2f} 天")
print(f"  H100 / A100 峰值比 {989.4 / 312:.2f}")

In [ ]:
print("\n第五步：回本")
print(f"  多训 10 万 GPU 小时、每次省 0.72 GPU 秒 → {breakeven_tasks(1e5 * 3600, 0.72):.0e} 次")

## 3. 代码里才看得到的

1. `expert_train_flops` 用的 V4-Flash 层数是 43、$h = 4096$、$f = 2048$；算出约 45.45 GFLOPs/token，和原文 45.4495 一致，说明原文这个数只计主干专家的三个投影。
2. `rmse` 只用两个留出点；把六个拟合点放进去，误差会小得多，这正是原文说的"拟合点不能代替留出点"。
3. 工期计算假设整个训练都用满同样多的卡；真实训练分阶段，卡数会变，原文说的"资源分配时序决定完成时间"就是这个意思。

## 4. 自己改一改

- 用 `alloc_exponents` 试 Kaplan 的分配：要让 $N \propto C^{0.73}$，$\alpha$ 和 $\beta$ 的比大概是多少？
- 把 DeepSeek-V3 的 2.664M H800 小时换算成 A100 等效小时。
- 把回本例子里每次节省从 0.72 秒降到 0.1 秒，看回本次数涨多少。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 装修预算分面积和质量，指数定怎么分；
> 模拟考贴得准，留出的才算数；
> 店开得越久，越值得多培训厨师；
> 人·天除以人数是工期，换卡先按峰值折；
> 截距是训练，斜率是服务，交点就是回本。